In [ ]:
import anthropic
import json

client = anthropic.Anthropic()
model = "claude-opus-4-6"

In [ ]:
def stream_conversation(messages, tools, fine_grained=False):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "tools": tools,
    }
    if fine_grained:
        params["betas"] = ["fine-grained-tool-streaming-2025-05-14"]

    with client.messages.stream(**params) as stream:
        for chunk in stream:
            if chunk.type == "content_block_delta":
                if hasattr(chunk.delta, "text"):
                    print(chunk.delta.text, end="", flush=True)
            if chunk.type == "input_json":
                print(chunk.partial_json, end="", flush=True)
        
        return stream.get_final_message()

In [ ]:
messages = [{"role": "user", "content": "What time is it right now?"}]

tools = [{
    "name": "get_current_datetime",
    "description": "Returns the current date and time.",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "Python strftime format string.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}]

response = stream_conversation(messages, tools=tools)
print("\n\nStop reason:", response.stop_reason)

In [ ]:
response = stream_conversation(messages, tools=tools, fine_grained=True)
print("\n\nStop reason:", response.stop_reason)